# Additional reproducibility checks
These checks exercise the data-to-loss boundary, shared backbone initialization, batched cached generation, and interrupted-training continuation. They use small models and temporary checkpoints. Their outputs validate software behavior and do not measure final translation quality.


In [1]:
from pathlib import Path
import json, os
here=Path.cwd().resolve()
ROOT=next((p for p in [here,*here.parents,here/'part 1'] if (p/'shared'/'implementation.ipynb').exists()),None)
assert ROOT is not None, 'Set the notebook working directory inside part 1.'
for cell in json.loads((ROOT/'shared'/'implementation.ipynb').read_text())['cells']:
    if cell['cell_type']=='code' and 'definitions' in cell.get('metadata',{}).get('tags',[]):
        exec(compile(cell['source'],'shared/implementation.ipynb','exec'),globals())


In [2]:
train=PairStore('train'); val=PairStore('validation'); tok=Tokenizer.from_file(str(ROOT/'shared'/'tokenizer_clean.json'))
indices=np.arange(32); x,y,lang,seps=collate(train,indices)
for row,i in enumerate(indices):
    ids,sep,g=train.get(i); n=len(ids)-1
    assert torch.equal(x[row,:n].cpu(),torch.tensor(np.array(ids[:-1],dtype=np.int64)))
    assert torch.all(y[row,:sep]==-100) and torch.all(y[row,n:]==-100)
    assert torch.equal(y[row,sep:n].cpu(),torch.tensor(np.array(ids[sep+1:],dtype=np.int64)))
    assert int(y[row,n-1])==EOS
assert int(x.ne(PAD).sum())==int(train.lengths[indices].sum())
assert int(y.ne(-100).sum())==int((train.lengths[indices]-train.seps[indices]).sum())
print('PASS: real-data shift, source/PAD masking, EOS supervision and both token counters.')
small=Config(d=32,heads=4,layers=2,hidden=128,dropout=0.,vocab=16000,context=256)
seed_all(42); baseline=Transformer(replace(small,variant=1))
common={n:p for n,p in baseline.named_parameters() if '.ffn.' not in n}
for i in range(2,6):
    seed_all(42); candidate=Transformer(replace(small,variant=i))
    assert all(torch.equal(p,dict(candidate.named_parameters())[n]) for n,p in common.items())
print('PASS: identical common-backbone initialization across all five variants.')


PASS: real-data shift, source/PAD masking, EOS supervision and both token counters.
PASS: identical common-backbone initialization across all five variants.


In [3]:
# Cache correctness with unequal prompt lengths and a padding gap.
for variant in range(1,6):
    model=Transformer(replace(small,variant=variant)).to(DEVICE).eval()
    prompts=torch.tensor([[BOS,9,10,SEP],[BOS,11,SEP,PAD]],device=DEVICE)
    lengths=torch.tensor([4,3],device=DEVICE); valid=prompts.ne(PAD)
    with torch.no_grad():
        logits,cache=model(prompts)
        for step in range(3):
            token=torch.tensor([[12+step],[15+step]],device=DEVICE)
            valid=torch.cat([valid,torch.ones((2,1),device=DEVICE,dtype=torch.bool)],1)
            cached,cache=model(token,cache=cache,valid=valid,positions=(lengths+step)[:,None])
            for row,length in enumerate(lengths.tolist()):
                tail=[12+j if row==0 else 15+j for j in range(step+1)]
                direct=torch.tensor([prompts[row,:length].tolist()+tail],device=DEVICE)
                expected,_=model(direct)
                assert torch.allclose(cached[row,0],expected[0,-1],atol=1e-5)
print('PASS: cached and uncached logits agree with unequal prompt lengths for all five models.')


PASS: cached and uncached logits agree with unequal prompt lengths for all five models.


In [4]:
# A deterministic CPU continuation check, including nonzero dropout and sparse routing.
import tempfile, shutil
previous_device=DEVICE; DEVICE=torch.device('cpu')
resume_cfg=Config(variant=3,d=32,heads=4,layers=2,hidden=128,dropout=.1,token_budget=30_000,batch_tokens=1024,batch_examples=16,eval_interval=10_000,val_examples=32,profile='resume-check')
original_evaluator=evaluate_nll
with tempfile.TemporaryDirectory(prefix='resume-check-',dir=ROOT/'.cache') as temporary:
    base=Path(temporary); uninterrupted=base/'uninterrupted'; resumed=base/'resumed'; resumed.mkdir()
    def capture_previous_checkpoint(*args,**kwargs):
        result=original_evaluator(*args,**kwargs)
        previous=uninterrupted/'resume.pt'
        if previous.exists() and not (resumed/'resume.pt').exists(): shutil.copyfile(previous,resumed/'resume.pt')
        return result
    try:
        evaluate_nll=capture_previous_checkpoint
        full,full_summary,_=training_run(resume_cfg,train,val,uninterrupted,resume_cfg.token_budget)
        evaluate_nll=original_evaluator
        assert (resumed/'resume.pt').exists()
        continued,continued_summary,_=training_run(resume_cfg,train,val,resumed,resume_cfg.token_budget)
        equal=all(torch.equal(v,continued.state_dict()[k]) for k,v in full.state_dict().items())
        assert equal,'Resumed training changed final weights.'
        assert full_summary['input_tokens']==continued_summary['input_tokens']
        assert full_summary['target_tokens']==continued_summary['target_tokens']
        print('PASS: interrupted and uninterrupted sparse training produce bitwise-identical final weights and token counters.')
    finally:
        evaluate_nll=original_evaluator; DEVICE=previous_device


validation {'nll': 9.53676599385787, 'ppl': 13860.051574735659, 'target_tokens': 735, 'nll_vi': 9.522598918855499, 'ppl_vi': 13665.079537272832, 'nll_ja': 9.541810727207423, 'ppl_ja': 13930.148500631192}


validation {'nll': 9.41075376264092, 'ppl': 12219.077790365445, 'target_tokens': 735, 'nll_vi': 9.404781361318005, 'ppl_vi': 12146.31804547941, 'nll_ja': 9.412880466433029, 'ppl_ja': 12245.091801681701}


model 3 | 29,699/29,699 input tokens | nll 9.5094 | 1s


validation {'nll': 9.359213563049732, 'ppl': 11605.258148450721, 'target_tokens': 735, 'nll_vi': 9.356250352810084, 'ppl_vi': 11570.92022903436, 'nll_ja': 9.36026872831957, 'ppl_ja': 11617.510076564238}


validation {'nll': 9.41075376264092, 'ppl': 12219.077790365445, 'target_tokens': 735, 'nll_vi': 9.404781361318005, 'ppl_vi': 12146.31804547941, 'nll_ja': 9.412880466433029, 'ppl_ja': 12245.091801681701}


model 3 | 29,699/29,699 input tokens | nll 9.4317 | 1s


validation {'nll': 9.359213563049732, 'ppl': 11605.258148450721, 'target_tokens': 735, 'nll_vi': 9.356250352810084, 'ppl_vi': 11570.92022903436, 'nll_ja': 9.36026872831957, 'ppl_ja': 11617.510076564238}


PASS: interrupted and uninterrupted sparse training produce bitwise-identical final weights and token counters.
